# FinRisk Radar — Notebook 03: RAG Pipeline

End-to-end walkthrough: SEC filing → chunks → embeddings → retrieval → Gemini answer.

In [ ]:
import sys
sys.path.append('..')

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../.env')

TICKER = 'AAPL'  # Change this to test other companies
print(f'Testing RAG pipeline for: {TICKER}')

## Step 1: Ingest SEC Filing

In [ ]:
from data.ingestion import ingest_sec_filings

print(f'Downloading SEC filings for {TICKER}...')
filings = ingest_sec_filings(TICKER, form_types=['10-K'])
print(f'Downloaded {len(filings)} filings')
for f in filings:
    print(f"  {f['form']} | {f['date']} | {f['path']}")

## Step 2: Text Extraction + Chunking

In [ ]:
from rag.ingestion import (
    extract_text_from_html, filter_relevant_sections, chunk_text
)

if filings:
    sample_path = filings[0]['path']
    raw_text = extract_text_from_html(Path(sample_path).read_text(errors='replace'))
    filtered  = filter_relevant_sections(raw_text)
    chunks    = chunk_text(filtered, TICKER, filings[0]['date'], '10-K')
    
    print(f'Raw text length: {len(raw_text):,} chars')
    print(f'Filtered length: {len(filtered):,} chars ({len(filtered)/len(raw_text):.0%} of original)')
    print(f'Number of chunks: {len(chunks)}')
    print(f'\nSample chunk (chunk #5):')
    print(chunks[5]['text'][:500])
    print(f'\nMetadata: {chunks[5]["metadata"]}')

## Step 3: Embed + Store in ChromaDB

In [ ]:
from rag.ingestion import get_embedding_model, ingest_chunks_to_chroma

print('Loading embedding model (all-MiniLM-L6-v2)...')
embed_model = get_embedding_model()

# Test embedding speed
import time
sample_texts = [c['text'] for c in chunks[:10]]
t0 = time.time()
embeddings = embed_model.embed_documents(sample_texts)
elapsed = time.time() - t0

print(f'Embedding 10 chunks: {elapsed:.2f}s ({elapsed/10*1000:.0f}ms per chunk)')
print(f'Embedding dimension: {len(embeddings[0])}')

# Ingest all chunks
print(f'\nIngesting {len(chunks)} chunks into ChromaDB...')
ingest_chunks_to_chroma(chunks, TICKER, embed_model)
print('Done!')

## Step 4: MMR Retrieval

In [ ]:
from rag.retriever import retrieve_mmr, retrieve_similarity

query = 'What are the main liquidity risks and cash flow concerns?'
print(f'Query: {query}\n')

# Compare MMR vs similarity
mmr_results  = retrieve_mmr(query, TICKER, k=5, embedding_model=embed_model)
sim_results  = retrieve_similarity(query, TICKER, k=5, embedding_model=embed_model)

print('=== MMR Results ===')
for i, r in enumerate(mmr_results, 1):
    print(f'[{i}] {r["source"]}')
    print(f'    {r["text"][:150]}...\n')

print('=== Similarity Results ===')
for i, r in enumerate(sim_results, 1):
    print(f'[{i}] {r["source"]} (score: {r.get("score", "N/A")})')
    print(f'    {r["text"][:150]}...\n')

## Step 5: Cross-Encoder Reranking

In [ ]:
from rag.retriever import retrieve

# Full pipeline: MMR(20) → rerank → top 5
final_chunks = retrieve(query, TICKER, k=5, use_reranker=True,
                        embedding_model=embed_model)

print('Final reranked chunks:')
for i, c in enumerate(final_chunks, 1):
    print(f'[{i}] rerank_score={c.get("rerank_score", "N/A"):.3f} | {c["source"]}')
    print(f'     {c["text"][:200]}\n')

## Step 6: Generate Answer with Gemini

In [ ]:
from rag.retriever import format_context
from rag.generator import generate, generate_structured

context = format_context(final_chunks)
print(f'Context length: {len(context):,} chars\n')

# Standard generation
result = generate(query, context, TICKER, risk_score=30.0)
print('=== Generated Answer ===')
print(result['answer'])
print(f'\nCitations used: {result["citations_found"]}')

## Step 7: Structured JSON Output

In [ ]:
import json

structured = generate_structured(query, context, TICKER, risk_score=30.0)
if structured['success']:
    print(json.dumps(structured['data'], indent=2))
else:
    print('Structured output failed:', structured.get('error'))

## Step 8: Run RAGAS Evaluation

In [ ]:
from evaluation.ragas_eval import build_eval_dataset

print('Generating synthetic QA dataset from filing chunks...')
eval_df = build_eval_dataset(TICKER, n_chunks=10)  # Start small
print(f'Generated {len(eval_df)} QA pairs')
print(eval_df[['question', 'ground_truth']].head(3).to_string())

In [ ]:
# Full RAGAS evaluation (takes ~2 minutes)
from evaluation.ragas_eval import run_ragas_eval

print('Running RAGAS evaluation (this calls Gemini API)...')
scores = run_ragas_eval(TICKER, sample_size=5)  # Small sample for notebook

print('\nRAGAS Results:')
for metric, val in scores.items():
    if isinstance(val, float):
        bar = '█' * int(val * 20)
        target = {'faithfulness': 0.85, 'context_recall': 0.80,
                  'answer_relevancy': 0.78, 'context_precision': 0.75}
        t = target.get(metric, 0)
        status = '✅' if val >= t else '❌'
        print(f'  {status} {metric:<25} {val:.4f}  {bar}')